# 14 — Enterprise: Multi-Tenant Deployment & Audit

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anulum/director-ai/blob/main/notebooks/14_enterprise_multi_tenant.ipynb)

Deploy Director-AI as a shared service across multiple teams or
customers with isolated facts, policies, and audit trails.

**What you'll learn:**
1. Tenant isolation architecture
2. Per-tenant policies and thresholds
3. Audit logging for compliance
4. REST API server deployment
5. gRPC server for low-latency
6. Docker and Kubernetes deployment

Requires `pip install director-ai[enterprise,server]`.

In [ ]:
!pip install -q director-ai

## 1. Tenant Isolation Architecture

```
┌─────────────┐  ┌─────────────┐  ┌─────────────┐
│  Tenant A    │  │  Tenant B    │  │  Tenant C    │
│  (Medical)   │  │  (Finance)   │  │  (Support)   │
├─────────────┤  ├─────────────┤  ├─────────────┤
│ Facts: HIPAA │  │ Facts: SEC   │  │ Facts: FAQ   │
│ Thresh: 0.8  │  │ Thresh: 0.75 │  │ Thresh: 0.5  │
│ NLI: Yes     │  │ NLI: Yes     │  │ NLI: No      │
│ Audit: Full  │  │ Audit: Full  │  │ Audit: Light │
└──────┬───────┘  └──────┬───────┘  └──────┬───────┘
       │                 │                 │
       └─────────────────┼─────────────────┘
                         │
              ┌──────────▼──────────┐
              │    Director-AI      │
              │  Shared NLI Model   │
              │  Shared Infra       │
              └─────────────────────┘
```

Each tenant gets:
- Isolated fact store (facts tagged with `tenant_id`)
- Custom thresholds and scoring policy
- Separate audit log stream
- API key scoping

In [ ]:
from director_ai.core import CoherenceScorer, GroundTruthStore

# Shared store with tenant isolation
store = GroundTruthStore()

# Tenant A: medical
store.add("dosage_ibu", "Ibuprofen max: 400mg q6h.", tenant_id="medical_corp")
store.add("dosage_acet", "Acetaminophen max: 1000mg q6h.", tenant_id="medical_corp")

# Tenant B: fintech
store.add(
    "rate",
    "Current APR: 5.99% for qualified applicants.",
    tenant_id="fintech_inc",
)
store.add("fees", "No annual fee. $35 late payment fee.", tenant_id="fintech_inc")

# Per-tenant scorers with different thresholds
scorer_medical = CoherenceScorer(threshold=0.8, ground_truth_store=store, use_nli=False)
scorer_fintech = CoherenceScorer(threshold=0.6, ground_truth_store=store, use_nli=False)

# Medical tenant query — only sees medical facts
ctx = store.retrieve_context("ibuprofen dosage", tenant_id="medical_corp")
print(f"Medical context: {ctx}")

# Fintech tenant query — only sees fintech facts
ctx = store.retrieve_context("interest rate", tenant_id="fintech_inc")
print(f"Fintech context: {ctx}")

## 2. REST API Server

Deploy as a FastAPI server with `pip install director-ai[server]`.

```bash
# Start server
director-ai serve --host 0.0.0.0 --port 8080

# Score a response
curl -X POST http://localhost:8080/v1/score \
  -H 'Content-Type: application/json' \
  -d '{
    "query": "What is the refund policy?",
    "response": "Full refund within 30 days.",
    "facts": {"refund": "30-day refund window."}
  }'

# Response:
# {
#   "approved": true,
#   "score": 0.872,
#   "h_logical": 0.12,
#   "h_factual": 0.08
# }
```

## 3. gRPC Server (Low Latency)

For latency-critical deployments, use the gRPC interface.

```bash
pip install director-ai[grpc]
director-ai serve --grpc --port 50051
```

```python
import grpc
from director_ai.director_pb2 import ScoreRequest
from director_ai.director_pb2_grpc import DirectorStub

channel = grpc.insecure_channel("localhost:50051")
stub = DirectorStub(channel)

response = stub.Score(ScoreRequest(
    query="What is the policy?",
    text="30-day refund.",
))
print(f"Score: {response.score}, approved: {response.approved}")
```

## 4. Docker Deployment

```dockerfile
# Dockerfile
FROM python:3.12-slim
RUN pip install director-ai[server,nli]
EXPOSE 8080
CMD ["director-ai", "serve", "--host", "0.0.0.0", "--port", "8080"]
```

```bash
docker build -t director-ai .
docker run -p 8080:8080 director-ai
```

For GPU inference:
```bash
docker build -f Dockerfile.gpu -t director-ai-gpu .
docker run --gpus all -p 8080:8080 director-ai-gpu
```

## 5. Kubernetes Deployment

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: director-ai
spec:
  replicas: 3
  selector:
    matchLabels:
      app: director-ai
  template:
    metadata:
      labels:
        app: director-ai
    spec:
      containers:
      - name: director-ai
        image: director-ai:latest  # build locally: docker build -t director-ai .
        ports:
        - containerPort: 8080
        env:
        - name: DIRECTOR_USE_NLI
          value: "true"
        - name: DIRECTOR_COHERENCE_THRESHOLD
          value: "0.7"
        resources:
          requests:
            memory: "2Gi"
            cpu: "1"
          limits:
            memory: "4Gi"
            cpu: "2"
        livenessProbe:
          httpGet:
            path: /health
            port: 8080
          initialDelaySeconds: 30
        readinessProbe:
          httpGet:
            path: /health
            port: 8080
          initialDelaySeconds: 10
---
apiVersion: v1
kind: Service
metadata:
  name: director-ai
spec:
  selector:
    app: director-ai
  ports:
  - port: 80
    targetPort: 8080
  type: ClusterIP
```


## 6. Monitoring & Metrics

Director-AI exposes Prometheus-compatible metrics at `/metrics`.

| Metric | Type | Description |
|--------|------|-------------|
| `director_reviews_total` | counter | Total reviews processed |
| `director_hallucinations_total` | counter | Responses rejected |
| `director_coherence_score` | histogram | Score distribution |
| `director_review_duration_seconds` | histogram | Latency per review |
| `director_streaming_halts_total` | counter | Stream halts |
| `director_cache_hits_total` | counter | Score cache hits |

OpenTelemetry tracing is also available with `pip install director-ai[otel]`.

## Summary

| Deployment | Install | Latency | Scale |
|------------|---------|---------|-------|
| Library (in-process) | `pip install director-ai` | <1ms | Single process |
| REST API | `[server]` | ~5ms | Horizontal |
| gRPC | `[grpc]` | ~2ms | Horizontal |
| Docker | `Dockerfile` | ~5ms | Container orchestration |
| Kubernetes | Helm/manifests | ~5ms | Auto-scaling |